# 프롬프트 단독 실험 — "풀이가 길수록 정확한가"

## 가설
> **모델이 풀이를 자세히 쓸수록 정확도가 올라간다.**

CoT에서 토큰 하나하나가 계산 기회입니다. 중간값을 안 적고 넘어가면 그만큼 암산해야 하고, 거기서 실수가 납니다.

## 근거 두 가지

**① 공개 데이터 SFT 실패 (8/19)**
NuminaMath 풀이는 OCR된 교과서 문체라 짧습니다(토큰 중앙값 387 vs RFT 434).
학습 후 `pass@32`가 0.8633 → 0.8500으로 **하락**했습니다.
모델이 "간결하게 쓰는 법"을 배우면서 추론이 상한 것으로 보입니다.

**② 프롬프트 다양화 실험 (8/18)**

| 프롬프트 | maj@8 | 파싱실패 |
|---|---|---|
| A (기존, **`concisely` 포함**) | 0.7100 | 1.62% |
| **B (검산 강조)** | **0.7267** | 2.21% |
| C (계획 먼저) | 0.7167 | 2.38% |
| **D (중간값 전부 명시)** | **0.7233** | **1.00%** |

`concisely`가 없는 B·D가 더 높았습니다. 그땐 "노이즈"로 넘겼는데, ①과 방향이 맞습니다.

## 이번에 하는 것
**B와 D를 각각 단독으로 n=32** 측정합니다. 4종을 섞었던 지난번과 달리, 하나씩 봅니다.

그리고 **평균 출력 길이도 같이 잽니다.** 길이가 길고 정확도가 높으면 가설이 확인됩니다.

## 기준선 (프롬프트 A, n=32)
| 지표 | 값 |
|---|---|
| maj@32 | **0.7400** |
| pass@32 | **0.8633** |
| 평균 득표율 | 0.726 |
| 파싱 실패율 | 1.70% |

## 소요
프롬프트 2종 x 300문제 x 32샘플. 출력이 길어져 **약 2~3시간** 예상.


---
## [1] 설정 ▶️

In [ ]:
N_SAMPLES  = 32        # 기준선과 동일
TEMP       = 0.8
MAX_TOKENS = 1400      # ★ 1024→1400. 자세히 쓰라고 시키므로 잘림 방지
VALID_N    = 300       # 절대 변경 금지
SEED       = 42        # 절대 변경 금지
MODEL_ID   = "Qwen/Qwen2.5-3B-Instruct"
print(f"{VALID_N}문제 x {N_SAMPLES}샘플 x 프롬프트 2종")

---
## [2] vLLM 설치 ⏭️

In [ ]:
!pip install -q -U vllm 2>&1 | tail -3

---
## [3] protobuf ⏭️
---
## ⛔ Restart Session → [1]부터
---

In [ ]:
!pip install -q -U "protobuf>=6.33.6,<7" 2>&1 | tail -2
import google.protobuf as p
print("protobuf", p.__version__); assert p.__version__.startswith("6.")

---
## [4] 데이터 ▶️ 기준선과 같은 300문제

In [ ]:
import glob, os, pandas as pd

def find_csv(must_have, must_not=()):
    for p in sorted(glob.glob("/kaggle/input/**/*.csv", recursive=True)):
        b = os.path.basename(p).lower()
        if all(k in b for k in must_have) and not any(k in b for k in must_not):
            return p

train = pd.read_csv(find_csv(["train"], must_not=["filtered","ids","leaderboard","test"]))
train = train[~train["id"].isin(set(pd.read_csv(find_csv(["filtered","ids"]))["id"]))].reset_index(drop=True)
assert len(train) == 16373

work = train.sample(VALID_N, random_state=SEED).reset_index(drop=True)
gold = work["answer"].tolist()
print(f"{len(work)}문제 | 첫 id: {work.iloc[0]['id']}  (train-004925 여야 함)")

---
## [5] 답 추출기 ▶️

In [ ]:
import re
from collections import Counter

def extract_boxed(text):
    """Return the raw content inside the LAST \\boxed{...}, brace-balanced."""
    idx = text.rfind('\\boxed')
    if idx == -1:
        return None
    i = idx + len('\\boxed')
    while i < len(text) and text[i] == ' ':
        i += 1
    if i >= len(text):
        return None
    if text[i] != '{':                       # bare form: \boxed 15
        m = re.match(r'-?[\d,]+', text[i:])
        return m.group(0) if m else None
    depth, start = 0, i + 1
    while i < len(text):
        if text[i] == '{':
            depth += 1
        elif text[i] == '}':
            depth -= 1
            if depth == 0:
                return text[start:i]
        i += 1
    return None

def to_int(s):
    """LaTeX/text -> python int, or None. Never uses float(), so huge ints survive."""
    if s is None:
        return None
    s = str(s).strip()
    s = s.replace('{,}', '').replace('{\\,}', '')          # LaTeX thousands separator
    s = re.sub(r'\\(?:text|mathrm|mbox|textbf|textrm)\s*\{([^{}]*)\}', r'\1', s)
    for junk in ['\\!', '\\,', '\\;', '\\:', '\\ ', '\\left', '\\right',
                 '\\$', '$', '%', '~', '^\\circ', '\\%']:
        s = s.replace(junk, '')
    s = s.replace(',', '').replace(' ', '').strip()
    s = re.sub(r'[a-zA-Z]+$', '', s)                       # trailing unit: 42cm -> 42
    while len(s) > 1 and s[0] == '(' and s[-1] == ')':     # (\frac{100}{4}) -> \frac{100}{4}
        s = s[1:-1].strip()
    s = s.rstrip('.')
    if not s:
        return None
    m = re.fullmatch(r'\\[dt]?frac\{([-+]?\d+)\}\{([-+]?\d+)\}', s)
    if m:
        a, b = int(m.group(1)), int(m.group(2))
        return a // b if b != 0 and a % b == 0 else None
    m = re.fullmatch(r'([-+]?\d+)/([-+]?\d+)', s)
    if m:
        a, b = int(m.group(1)), int(m.group(2))
        return a // b if b != 0 and a % b == 0 else None
    m = re.fullmatch(r'([-+]?\d+)(?:\\times|\\cdot)10\^\{?(\d+)\}?', s)
    if m:
        return int(m.group(1)) * 10 ** int(m.group(2))
    if re.fullmatch(r'[-+]?\d+', s):
        return int(s)
    m = re.fullmatch(r'([-+]?\d+)\.0*', s)
    if m:
        return int(m.group(1))
    return None

def last_int(text):
    for c in reversed(re.findall(r'-?\d[\d,]*', text)):
        v = to_int(c)
        if v is not None:
            return v
    return None

def parse_answer(text):
    """None means 'this sample produced no usable integer' -> dropped from voting."""
    raw = extract_boxed(text)
    if raw is not None:
        return to_int(raw)          # boxed present but unparseable -> None, do NOT guess
    m = re.findall(r'(?:answer|Answer|ANSWER)\s*(?:is|:|=)+\s*\$?(-?[\d,]+)', text)
    if m:
        v = to_int(m[-1])
        if v is not None:
            return v
    return last_int(text)

def majority_vote(values, fallback=0):
    vals = [v for v in values if v is not None]
    if not vals:
        return fallback
    return Counter(vals).most_common(1)[0][0]

_c=[(r"\boxed{132}",132),(r"\boxed{-2,025,078}",-2025078),(r"\boxed{\frac{7}{2}}",None)]
print("parser FAILURES:", sum(parse_answer(t)!=w for t,w in _c), "/", len(_c))

---
## [6] 프롬프트 2종 ▶️

지난번 4종 중 **A보다 높았던 두 개**만 가져옵니다. 둘 다 `concisely`가 없습니다.

| | 전략 | 노리는 것 |
|---|---|---|
| **B** | 검산 강조 | 산술 실수를 스스로 잡게 |
| **D** | 중간값 전부 명시 | 암산을 막아 실수 자체를 줄임 |

지난번과 **한 글자도 다르지 않게** 유지합니다. 그래야 n=8 결과와 이어집니다.

In [ ]:
from transformers import AutoTokenizer

BOX = " End your response with the final integer inside \\boxed{}."

PROMPTS = {
 "B_verify":
    "You are a careful mathematician. Solve the problem, then verify every arithmetic "
    "step before committing to an answer. If a check fails, redo that step. "
    "The final answer is ALWAYS a single integer." + BOX,
 "D_explicit":
    "Solve the problem showing every intermediate value explicitly. Never skip a "
    "calculation or do arithmetic mentally. The final answer is ALWAYS a single integer." + BOX,
}

tok = AutoTokenizer.from_pretrained(MODEL_ID)
def build(sys_txt):
    return [tok.apply_chat_template(
        [{"role":"system","content":sys_txt},{"role":"user","content":q}],
        tokenize=False, add_generation_prompt=True) for q in work["question"]]

print(f"프롬프트 {len(PROMPTS)}종:", list(PROMPTS))

---
## [7] 모델 로드 + 채점 함수 ▶️

### 이번에 추가로 재는 것: 평균 출력 길이
가설이 "길수록 정확하다"이므로, **길이와 정확도가 같이 움직이는지**를 봐야 합니다.

- 길이 ↑ + 정확도 ↑ → 가설 확인
- 길이 ↑ + 정확도 그대로 → 길이는 원인이 아님
- 길이 ↑ + **잘림 비율 ↑** → `MAX_TOKENS` 부족. 결과 왜곡 가능

`finish_reason != "stop"`이면 토큰 한도에 걸려 잘린 것입니다. 잘린 출력엔 `\boxed{}`가 없어 파싱 실패가 됩니다.

In [ ]:
import time, numpy as np
from collections import Counter
from vllm import LLM, SamplingParams

llm = LLM(model=MODEL_ID, dtype="half", max_model_len=4096,
          gpu_memory_utilization=0.90, tensor_parallel_size=1,
          seed=SEED, trust_remote_code=True)
sp = SamplingParams(n=N_SAMPLES, temperature=TEMP, top_p=0.95,
                    max_tokens=MAX_TOKENS, seed=SEED)

def score(outs, label):
    maj, psk, shares, fails, toks, trunc = [], [], [], 0, [], 0
    for o, g in zip(outs, gold):
        vals = []
        for c in o.outputs:
            toks.append(len(c.token_ids))
            trunc += (c.finish_reason != "stop")
            vals.append(parse_answer(c.text))
        fails += sum(v is None for v in vals)
        valid = [v for v in vals if v is not None]
        cnt = Counter(valid)
        m = cnt.most_common(1)[0][0] if cnt else 0
        maj.append(int(m) == int(g))
        psk.append(any(int(v) == int(g) for v in valid))
        shares.append(cnt[m]/len(vals) if cnt else 0)
    n = len(outs) * N_SAMPLES
    r = dict(label=label, maj=np.mean(maj), pas=np.mean(psk), share=np.mean(shares),
             fail=fails/n, tok=np.mean(toks), trunc=trunc/n, maj_list=maj)
    print(f"[{label:<11}] maj={r['maj']:.4f}  pass={r['pas']:.4f}  득표율={r['share']:.3f}  "
          f"파싱실패={r['fail']:.2%}  평균토큰={r['tok']:.0f}  잘림={r['trunc']:.2%}")
    return r

results = {}
print("로드 완료")

---
## [8] B 프롬프트 측정 ▶️ 약 1~1.5시간

In [ ]:
t0 = time.time()
outs_b = llm.generate(build(PROMPTS["B_verify"]), sp)
print(f"생성 {(time.time()-t0)/60:.1f}분")
results["B_verify"] = score(outs_b, "B_verify")

---
## [9] D 프롬프트 측정 ▶️ 약 1~1.5시간

In [ ]:
t0 = time.time()
outs_d = llm.generate(build(PROMPTS["D_explicit"]), sp)
print(f"생성 {(time.time()-t0)/60:.1f}분")
results["D_explicit"] = score(outs_d, "D_explicit")

---
## [10] 결론 ▶️ (GPU 미사용)

### 판정
| 결과 | 다음 수 |
|---|---|
| maj@32 > 0.7400 (+1%p 이상) | **채택 → 리더보드 제출** |
| +0~1%p | 애매. McNemar로 판단 |
| 하락 | 기존 프롬프트 유지, 8/31 준비로 전환 |

### 가설 확인
평균 토큰이 기준선보다 길고 정확도도 높으면 **"자세한 풀이가 정확도를 올린다"**가 확인됩니다.
그러면 공개 데이터 SFT 실패 원인도 같은 설명으로 닫힙니다.

In [ ]:
import math

BASE = dict(maj=0.7400, pas=0.8633, share=0.726, fail=0.0170)
print("=" * 76)
print(f"{'기준선 A':<13} maj={BASE['maj']:.4f}  pass={BASE['pas']:.4f}  "
      f"득표율={BASE['share']:.3f}  파싱실패={BASE['fail']:.2%}   (MAX_TOKENS=1024)")
print("=" * 76)

for k, r in results.items():
    d = (r["maj"] - BASE["maj"]) * 100
    print(f"{k:<13} maj={r['maj']:.4f} ({d:+.2f}%p)  pass={r['pas']:.4f} "
          f"({(r['pas']-BASE['pas'])*100:+.2f}%p)  평균토큰={r['tok']:.0f}")

best = max(results.values(), key=lambda r: r["maj"])
print(f"\n최고: {best['label']}  maj@32 = {best['maj']:.4f}  ({(best['maj']-BASE['maj'])*100:+.2f}%p)")
print(f"리더보드 환산(x0.63) → 예상 {0.78580 + (best['maj']-BASE['maj'])*0.63:.5f}")

print("\n※ 참고: 기준선 A는 MAX_TOKENS=1024, 이번은 1400이라 완전한 동일 조건은 아님")
print("   (자세히 쓰라는 지시가 잘리지 않게 하려는 의도적 변경)")